# L05 · 复数（complex number）几何本质——频谱（spectrum）分析语境版

**学习目标**
1. 理解复数 `a+bj` 的直角坐标（rectangular coordinates / Cartesian coordinates）与极坐标（polar coordinates）表示
2. 实现 `magnitude_phase(z)` 返回 `(模（magnitude，r）, 相位（phase，φ）)` 元组
3. 理解 FFT 输出为什么是复数：模 = 该频率（frequency）有多响，相位 = 时间对齐
4. 为 L06 欧拉公式（Euler's formula） `e^{jθ} = cos(θ) + j·sin(θ)` 打好基础

**来自 L04 的记忆**：`sinusoid` 的振幅（amplitude） A 和相位 φ，
在 FFT 输出里分别对应一个复数的**模**和**相位**。
本课让你亲手从复数中读出这两个量。

**Aurora 连接**：`magnitude_spectrogram` = 对 FFT 每个频点（frequency bin）的复数取模；
相位在声码器（vocoder）和波形重建（L45）里至关重要。

← **上一课**　[L04 · 正弦波三要素](L04_trig.ipynb)

> 上节课学习了 **正弦波三要素**：频率决定音高、振幅决定响度、相位决定起点，亲手实现。  
> 本课将探讨 **复数几何本质**。

## 本课剧情：给频率分量办身份证

FFT 输出的每个频点是一个复数 `X[k] = a + bj`。
这个复数的两种等价描述：

| 表示 | 分量 | 物理含义 |
|---|---|---|
| 直角坐标 | 实部 a, 虚部 b | 余弦分量、正弦分量 |
| 极坐标 | 模 r = √(a²+b²), 相位 θ = arctan2(b,a) | **响度**, **时间对齐** |

极坐标在频谱分析里更自然：
- **模 r**：这个频率的能量有多强（谱图（spectrogram）上的颜色亮度）
- **相位 θ**：这个正弦波（sinusoid / sine wave）的起始角度（声码器用它来重建波形）

本课实现 `magnitude_phase(z)`，把任意复数拆成 `(r, θ)` 两个浮点数。

## 1. 复数的基本操作（Python 用 `j` 表示虚数单位）

`a + bj` 有两种等价描述：直角坐标 `(a, b)` 和极坐标 `(r, θ)`。转换关系：`r = √(a²+b²)`，`θ = arctan2(b, a)`；逆变换：`a = r·cos(θ)`，`b = r·sin(θ)`。

极坐标在频谱分析里比直角坐标更自然：模 `r` 直接对应"这个频率有多响"，相位 `θ` 直接对应"这个正弦波的起始时刻"。FFT 每个频点输出一个复数，取模得到 magnitude spectrogram（各频率的响度），取相位得到时间对齐信息——相位重建是声码器的核心操作。两者相互独立：改变响度不影响时序，改变起始时刻不影响响度。

## 实验入口：角度、坐标和旋转

角度 θ 对应的单位复数是 `cos(θ) + sin(θ)j`：实部是水平分量，虚部是垂直分量，模恒为 1。下面遍历一圈，验证极坐标与直角坐标的互换关系。

In [ ]:
import numpy as np
z = 3 + 4j
print('实部:', z.real, ' 虚部:', z.imag)
print('模 |z| =', abs(z), ' (=√(3²+4²)=5)')
print('相位(弧度) =', np.angle(z))

## 动手观察：复数就是"旋转的位置"

观察下表的 `real` 和 `imag` 列：角度每增加 π/2，坐标在单位圆（unit circle）上逆时针移一步。`magnitude` 列始终为 1，改变的只有 `phase`。

In [ ]:
import numpy as np

angles = np.array([0, np.pi/2, np.pi, 3*np.pi/2])
z = np.exp(1j * angles)

print('角度 =', np.round(angles, 3))
print('实部 cos =', np.round(z.real, 3))
print('虚部 sin =', np.round(z.imag, 3))
print('复数 z =', np.round(z, 3))


## 代码实验：旋转一整圈

从 0 到 2π 均匀取 8 个角度，打印每个点的实部、虚部、模和相位，验证单位圆上模始终为 1、相位就是角度本身。

In [ ]:
import numpy as np

for k in range(9):
    theta = 2 * np.pi * k / 8
    z = np.exp(1j * theta)
    print(f'k={k} theta={theta:.2f} -> ({z.real:+.3f}, {z.imag:+.3f})')


## 2. ✏️ 实现 `magnitude_phase(z)` 返回 `(模, 相位)`

**推理路线**：
1. 模是复数到原点的距离：`r = √(real² + imag²)`，Python 内置 `abs(z)` 直接返回此值，对标量和 numpy 数组均适用。
2. 相位是与 x 轴正半轴的夹角：`θ = arctan2(imag, real)`，`np.angle(z)` 封装了此计算，输出范围 `(-π, π]`，单位弧度（radian）。
3. 返回元组 `(magnitude, phase)` 是因为 FFT 后处理通常同时需要两者，比分开调用两次更高效。

**参考输入输出**：`z = 3 + 4j` → 模 = 5.0，相位 ≈ 0.9273 弧度（约 53°，即 arctan(4/3)）

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


### 写代码前，先把变量表补完整

写 `magnitude_phase` 前明确三件事：
- 输入：复数 `z`（Python 或 numpy 复数均可）
- 关键步骤：`abs(z)` 求模；`np.angle(z)` 求相位，范围 (−π, π]，单位弧度
- 返回：元组 `(magnitude, phase)`，两者均为浮点数

### 📐 手动实现规则（先自己做，再看提示）

**第一步（必做）**：仅用下列工具填写 stub：
- `z.real`，`z.imag` — 读取实部、虚部
- `np.sqrt()` — 开根号
- `np.arctan2(b, a)` — 反正切，返回 (-π, π]

**不允许** 在 stub 里直接调用 `abs(z)` 或 `np.angle(z)`（那等于零实现）。

通过断言后，在函数末尾加注释：`# 等价于: return abs(z), np.angle(z)` — 对比两种写法结果是否完全一致。


In [ ]:
def magnitude_phase(z):
    # ✏️ TODO: 仅用 z.real, z.imag, np.sqrt, np.arctan2 实现（不允许直接调 abs/np.angle）
    raise NotImplementedError("请实现 magnitude_phase(z)：用 z.real, z.imag, np.sqrt, np.arctan2")


In [ ]:
import numpy as np
try:
    mag, ph = magnitude_phase(3 + 4j)
    print('模 =', mag, ' 相位 =', round(ph, 4))
    assert abs(mag - 5.0) < 1e-9, f"模应为 5.0，实际得到 {mag}"
    assert abs(ph - np.arctan2(4, 3)) < 1e-9, f"相位应为 {np.arctan2(4,3):.4f}，实际得到 {ph}"
    print('✅ 通过：你能从复数读出模和相位了。')
except NotImplementedError as e:
    print(f'⚠️ 还未实现：{e}')
    print('  请回到上方 magnitude_phase 函数，用 z.real/z.imag/np.sqrt/np.arctan2 填写实现。')


**🔗 Aurora 连接**：`magnitude_spectrogram` 取的就是 FFT 复数结果的模；相位在声码器/重建里很关键。下一课：把复数和三角连起来——欧拉公式。

In [ ]:
for k in range(8):
    theta = 2*np.pi*k/8
    z = np.exp(1j*theta)
    radius = abs(z)
    print(f'k={k} | z=({z.real:+.2f}, {z.imag:+.2f}) | 半径={radius:.2f}')
print('所有点半径都接近 1，说明它们都在单位圆上。')


## 参数实验：单位圆上 8 点的模与相位

构造 `z = np.exp(1j * np.linspace(0, 2*np.pi, 8))`，打印每个点的模和相位，确认模恒为 1.0，相位均匀分布在 0 到约 5.5 弧度（即 0, π/4, π/2, …, 7π/4）。

再把 `3 + 4j` 换成 `3 - 4j`，观察相位从 +0.9273 变为 -0.9273：虚部取反即相位取反，模不变——验证模与相位相互独立。

In [ ]:
import numpy as np

# 实验 A：单位圆上 8 点的模与相位
z8 = np.exp(1j * np.linspace(0, 2 * np.pi, 8, endpoint=False))
print('单位圆 8 点（endpoint=False，避免重复 0/2π）')
print(f'  模：{np.abs(z8).round(4)}')  # 全部应为 1.0
print(f'  相位（弧度）：{np.angle(z8).round(4)}')  # 0, π/4, π/2, ..., 7π/4
assert np.allclose(np.abs(z8), 1.0), '模应全部为 1.0'
print('✅ 模恒为 1，相位均匀分布')

# 实验 B：虚部取反 → 相位取反，模不变
z_pos = 3 + 4j
z_neg = 3 - 4j
print(f'\n3+4j：模={abs(z_pos):.4f}，相位={np.angle(z_pos):.4f}')
print(f'3-4j：模={abs(z_neg):.4f}，相位={np.angle(z_neg):.4f}')
assert abs(abs(z_pos) - abs(z_neg)) < 1e-9, '模应相等'
assert abs(np.angle(z_pos) + np.angle(z_neg)) < 1e-9, '相位应互为相反数'
print('✅ 虚部取反 = 相位取反，模不变')


## FFT 输出预览：每个频点都是一个复数

不用理解推导——只看输出的数据类型和值，建立「FFT 输出 = 复数数组」的直觉。

In [ ]:
import numpy as np

# 把 L04 的和弦信号做 FFT（Module 5 会从零实现；这里只看输出形状）
sr, dur = 200, 1.0
t = np.arange(round(dur * sr)) / sr
chord = (np.sin(2*np.pi*20*t) +
         np.sin(2*np.pi*32*t) +
         np.sin(2*np.pi*40*t)) / 3  # 三个频率

X = np.fft.rfft(chord)  # FFT 输出：复数数组
freqs = np.fft.rfftfreq(len(chord), 1/sr)

print(f'输入：{chord.shape}  float64')
print(f'输出：{X.shape}  {X.dtype}  ← 复数！')
print()

# 找三个峰值频率，打印对应复数
magnitudes = np.abs(X)
top3 = np.argsort(magnitudes)[-3:][::-1]
print(f'  频率 (Hz)  |  复数 X[k]              |  模（响度）|  相位（弧度）')
print('-' * 70)
for idx in sorted(top3):
    z = X[idx]
    mag, phase = abs(z), np.angle(z)
    print(f'  {freqs[idx]:>8.1f}  |  {z.real:+.2f} + {z.imag:+.2f}j  |  {mag:.2f}        |  {phase:.3f}')

print()
print('模 ≈ 33（振幅 1/3 × 采样点数/2 = 1/3 × 100 ≈ 33）；归一化原因（Parseval 定理）将在 L40 推导，现在接受这个数字即可。')
print('相位 ≈ -π/2，因为 sin 波在 t=0 时值为 0，cos 相位领先 π/2。')

## 本课收束

现在你能用 `magnitude_phase(z)` 从任意复数读出模（`abs(z)`）
和相位（`np.angle(z)`）。

**关键连接**：
- L04 的 `sinusoid(t, A=r, f=f0, phi=θ)` ↔ FFT 输出复数 `X[k]` 的 `(模=r, 相位=θ)`
- `magnitude_spectrogram = |FFT 输出|`（逐元素取模，L40 实现）
- 相位谱（phase spectrum）在波形重建（L45 STFT 逆变换）中不可或缺

**下一课 L06**：欧拉公式 `e^{jθ} = cos(θ) + j·sin(θ)`——
把今天的极坐标和上节课的正弦波直接连起来，是理解 DFT 矩阵（L37）的最短路径。

---

→ **下一课**　[L06 · 欧拉公式 e^{iθ}=cosθ+isinθ](L06_euler.ipynb)

> 下节课将学习 **欧拉公式 e^{iθ}=cosθ+isinθ**：旋转因子是 FFT 的命根子。